In [27]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss, accuracy_score, f1_score

from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)

In [28]:
class CFG:
    model_name = "/kaggle/input/models/mohammedhamdan/mdeberta-v3-base/transformers/default/1/mdeberta-v3-base"
    max_len = 1024
    batch_size = 8
    epochs = 3
    lr = 2e-5
    weight_decay = 0.01
    num_labels = 2
    num_folds = 5
    seed = 42
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [29]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

In [30]:
LABEL2ID = {
    "model_a": 0,
    "model_b": 1,
}

ID2LABEL = {v: k for k, v in LABEL2ID.items()}

In [31]:
def build_input_text(prompt, response_a, response_b):
    text = (
        f"Prompt: {prompt}\n\n"
        f"Response A: {response_a}\n\n"
        f"Response B: {response_b}"
    )
    return text

In [33]:
def load_data(train_path):
    df = pd.read_parquet(train_path).copy()

    # 构造输入文本
    df["input_text"] = df.apply(
        lambda x: build_input_text(x["prompt"], x["response_a"], x["response_b"]),
        axis=1
    )

    # 标签映射
    df["label"] = df["winner"].map(LABEL2ID)

    return df

In [34]:
def get_tokenizer():
    tokenizer = AutoTokenizer.from_pretrained(CFG.model_name)
    return tokenizer

In [35]:
class ArenaDataset(Dataset):
    def __init__(self, df, tokenizer, max_len):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        encoded = self.tokenizer(
            row["input_text"],
            truncation=True,
            max_length=self.max_len,
            padding="max_length",
            return_tensors="pt"
        )

        item = {
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0),
            "labels": torch.tensor(row["label"], dtype=torch.long),
        }

        if "token_type_ids" in encoded:
            item["token_type_ids"] = encoded["token_type_ids"].squeeze(0)

        return item

In [37]:
def get_model():
    config = AutoConfig.from_pretrained(CFG.model_name)
    config.num_labels = CFG.num_labels

    model = AutoModelForSequenceClassification.from_pretrained(
        CFG.model_name,
        config=config
    )
    return model

In [38]:
def train_one_epoch(model, loader, optimizer, scheduler, device):
    model.train()
    total_loss = 0.0

    for batch in tqdm(loader, total=len(loader)):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        token_type_ids = batch.get("token_type_ids")
        if token_type_ids is not None:
            token_type_ids = token_type_ids.to(device)
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids,
                labels=labels
            )
        else:
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

        loss = outputs.loss
        loss.backward()

        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

        total_loss += loss.item()

    return total_loss / len(loader)

In [39]:
@torch.no_grad()
def valid_one_epoch(model, loader, device):
    model.eval()
    total_loss = 0.0

    all_labels = []
    all_probs = []

    for batch in tqdm(loader, total=len(loader)):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        token_type_ids = batch.get("token_type_ids")
        if token_type_ids is not None:
            token_type_ids = token_type_ids.to(device)
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids,
                labels=labels
            )
        else:
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

        loss = outputs.loss
        logits = outputs.logits
        probs = torch.softmax(logits, dim=-1)

        total_loss += loss.item()
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

    all_labels = np.array(all_labels)
    all_probs = np.array(all_probs)
    preds = all_probs.argmax(axis=1)

    metrics = {
        "valid_loss": total_loss / len(loader),
        "log_loss": log_loss(all_labels, all_probs),
        "accuracy": accuracy_score(all_labels, preds),
        "macro_f1": f1_score(all_labels, preds, average="macro")
    }

    return metrics, all_probs, all_labels

In [41]:
def run_cv(train_path):
    set_seed(CFG.seed)

    df = load_data(train_path)
    tokenizer = get_tokenizer()

    skf = StratifiedKFold(
        n_splits=CFG.num_folds,
        shuffle=True,
        random_state=CFG.seed
    )

    all_fold_metrics = []

    for fold, (train_idx, valid_idx) in enumerate(skf.split(df, df["label"])):
        print(f"\n========== Fold {fold} ==========")

        train_df = df.iloc[train_idx].reset_index(drop=True)
        valid_df = df.iloc[valid_idx].reset_index(drop=True)

        train_dataset = ArenaDataset(train_df, tokenizer, CFG.max_len)
        valid_dataset = ArenaDataset(valid_df, tokenizer, CFG.max_len)

        train_loader = DataLoader(
            train_dataset,
            batch_size=CFG.batch_size,
            shuffle=True,
            num_workers=2
        )

        valid_loader = DataLoader(
            valid_dataset,
            batch_size=CFG.batch_size,
            shuffle=False,
            num_workers=2
        )

        model = get_model().to(CFG.device)

        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=CFG.lr,
            weight_decay=CFG.weight_decay
        )

        num_train_steps = len(train_loader) * CFG.epochs
        scheduler = get_linear_schedule_with_warmup(
            optimizer,
            num_warmup_steps=int(0.1 * num_train_steps),
            num_training_steps=num_train_steps
        )

        best_logloss = 1e9
        best_metrics = None

        for epoch in range(CFG.epochs):
            print(f"\n--- Epoch {epoch + 1} ---")

            train_loss = train_one_epoch(model, train_loader, optimizer, scheduler, CFG.device)
            metrics, _, _ = valid_one_epoch(model, valid_loader, CFG.device)

            print(f"train_loss: {train_loss:.4f}")
            print(f"valid_loss: {metrics['valid_loss']:.4f}")
            print(f"log_loss:   {metrics['log_loss']:.4f}")
            print(f"accuracy:   {metrics['accuracy']:.4f}")
            print(f"macro_f1:   {metrics['macro_f1']:.4f}")

            if metrics["log_loss"] < best_logloss:
                best_logloss = metrics["log_loss"]
                best_metrics = metrics.copy()
                torch.save(model.state_dict(), f"baseline_fold{fold}.pth")

        all_fold_metrics.append(best_metrics)

    print("\n========== CV Result ==========")
    for i, m in enumerate(all_fold_metrics):
        print(f"Fold {i}: {m}")

    mean_logloss = np.mean([m["log_loss"] for m in all_fold_metrics])
    mean_acc = np.mean([m["accuracy"] for m in all_fold_metrics])
    mean_f1 = np.mean([m["macro_f1"] for m in all_fold_metrics])

    print(f"\nMean Log Loss: {mean_logloss:.4f}")
    print(f"Mean Accuracy: {mean_acc:.4f}")
    print(f"Mean Macro F1: {mean_f1:.4f}")

In [43]:
if __name__ == "__main__":
    train_path = "/kaggle/input/datasets/ai2334135131/train-data/train.parquet"
    run_cv(train_path)

The tokenizer you are loading from '/kaggle/input/models/mohammedhamdan/mdeberta-v3-base/transformers/default/1/mdeberta-v3-base' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/datasets/ai2334135131/train-data/test.parquet'